In [4]:
from __future__ import annotations

import os
import re
import time
from dataclasses import dataclass
from typing import List, Tuple

from qdrant_client import QdrantClient


# =========================
# CONFIG
# =========================
@dataclass
class Config:
    qdrant_url: str = "http://100.65.71.50:6333/"
    timeout: int = 120

config = Config()


# =========================
# Qdrant helpers
# =========================
def init_qdrant() -> QdrantClient:
    url = os.getenv("QDRANT_URL", config.qdrant_url).rstrip("/")
    api_key = os.getenv("QDRANT_API_KEY")
    return QdrantClient(url=url, api_key=api_key, timeout=config.timeout)

def wait_for_qdrant(client: QdrantClient, retries: int = 5, delay: int = 3) -> None:
    for attempt in range(1, retries + 1):
        try:
            client.get_collections()
            return
        except Exception as exc:
            if attempt == retries:
                raise SystemExit(
                    f"Cannot reach Qdrant at attempt {attempt}/{retries}: {exc}\n"
                    "Please ensure Qdrant is running and QDRANT_URL is correct."
                )
            print(f"Qdrant not reachable (attempt {attempt}/{retries}): {exc}. Retrying in {delay}s...")
            time.sleep(delay)

def list_collections(client: QdrantClient) -> List[str]:
    cols = client.get_collections()
    return [c.name for c in cols.collections]

def delete_collection_with_retry(client: QdrantClient, name: str, retries: int = 3, delay: int = 2) -> None:
    for attempt in range(1, retries + 1):
        try:
            client.delete_collection(collection_name=name)
            return
        except Exception as exc:
            if attempt == retries:
                raise
            print(f"[WARN] delete failed {name} (attempt {attempt}/{retries}): {exc}. Retrying in {delay}s...")
            time.sleep(delay)


# =========================
# Delete all collections with configurable suffix (v2/v3/...)
# =========================
def delete_all_collections_suffix(*, version: str = "v2", dry_run: bool = True) -> Tuple[List[str], List[Tuple[str, str]]]:
    """
    version: "v2" / "v3" / ... (hoặc bạn truyền "2" / "3" cũng được)
    dry_run=True  -> chỉ in danh sách sẽ xóa
    dry_run=False -> xóa thật

    Match các collection có tên kết thúc bằng:
      - 'vN'
      - '_vN', '-vN', '.vN'
      - case-insensitive
    """
    ver = str(version).strip().lower()
    if ver.isdigit():
        ver = f"v{ver}"
    if not re.fullmatch(r"v\d+", ver):
        raise ValueError(f"version phải dạng 'v2'/'v3'/... hoặc '2'/'3'/..., nhận: {version!r}")

    client = init_qdrant()
    wait_for_qdrant(client)

    all_names = list_collections(client)

    # match đuôi vN (cho phép trước đó có _, -, . hoặc không có)
    rx = re.compile(rf"(?:[_\-.]?{re.escape(ver)})$", re.IGNORECASE)

    targets = sorted([n for n in all_names if rx.search(n)])

    print(f"[INFO] Total collections: {len(all_names)}")
    print(f"[INFO] Target suffix: {ver}")
    print(f"[INFO] Matched: {len(targets)}")
    for n in targets[:50]:
        print(" -", n)
    if len(targets) > 50:
        print(f" ... (+{len(targets)-50} more)")

    if dry_run:
        print("[DRY-RUN] No collection deleted. Set dry_run=False to delete.")
        return targets, []

    errors: List[Tuple[str, str]] = []
    for name in targets:
        try:
            delete_collection_with_retry(client, name)
            print(f"[OK] Deleted: {name}")
        except Exception as e:
            err = repr(e)
            print(f"[ERR] Failed delete: {name} -> {err}")
            errors.append((name, err))

    print(f"[DONE] Deleted {len(targets) - len(errors)}/{len(targets)} collections. Errors={len(errors)}")
    return targets, errors


# ====== RUN ======
# xem trước:
targets, errors = delete_all_collections_suffix(version="v2", dry_run=False)

# xóa thật (bỏ comment):
# targets, errors = delete_all_collections_suffix(version="v3", dry_run=False)


[INFO] Total collections: 214
[INFO] Target suffix: v2
[INFO] Matched: 60
 - htlt_10_general_v2
 - htlt_11_general_v2
 - htlt_12_general_v2
 - htlt_1_general_v2
 - htlt_2_general_v2
 - htlt_3_general_v2
 - htlt_4_general_v2
 - htlt_5_general_v2
 - htlt_6_general_v2
 - htlt_7_general_v2
 - htlt_8_general_v2
 - htlt_9_general_v2
 - sd_10_general_v2
 - sd_11_general_v2
 - sd_12_general_v2
 - sd_1_general_v2
 - sd_2_general_v2
 - sd_3_general_v2
 - sd_4_general_v2
 - sd_5_general_v2
 - sd_6_general_v2
 - sd_7_general_v2
 - sd_8_general_v2
 - sd_9_general_v2
 - ttnt_10_LT_v2
 - ttnt_10_TH_v2
 - ttnt_11_LT_v2
 - ttnt_11_TH_v2
 - ttnt_12_LT_v2
 - ttnt_12_TH_v2
 - ttnt_1_LT_v2
 - ttnt_1_TH_v2
 - ttnt_2_LT_v2
 - ttnt_2_TH_v2
 - ttnt_3_LT_v2
 - ttnt_3_TH_v2
 - ttnt_4_LT_v2
 - ttnt_4_TH_v2
 - ttnt_5_LT_v2
 - ttnt_5_TH_v2
 - ttnt_6_LT_v2
 - ttnt_6_TH_v2
 - ttnt_7_LT_v2
 - ttnt_7_TH_v2
 - ttnt_8_LT_v2
 - ttnt_8_TH_v2
 - ttnt_9_LT_v2
 - ttnt_9_TH_v2
 - vhnt_10_general_v2
 - vhnt_11_general_v2
 ... (